In [1]:
library(ggplot2)
library(data.table)
library(stringr)
theme_set(theme_bw())

In [2]:
tools = c('singlem', 'sylph')
d1 = data.table(tool = tools, sample = "SRR8648366")

In [3]:
readit = function(tool, sample){
    to_read = paste0('output_',tool,'/', tool, '/',sample,'.profile')
    return(fread(to_read))
}
d2 = d1[, readit(tool, sample)[, c("coverage", "taxonomy")], by=list(tool, sample)]
d2[, relabu := coverage / sum(coverage, na.rm = TRUE), by = list(tool, sample)]
d2[, taxonomy := gsub("; ", ";", gsub("Root; ", "", taxonomy))]

In [4]:
m = dcast(d2, sample + taxonomy ~ tool, value.var = c("relabu", "coverage"), fill = 0)
setnames(m, names(m), gsub("_", "__", names(m)))

In [17]:
format_table <- function(m) {
    cov_cols <- grep("^coverage_|_coverage$", names(m), value = TRUE)
    mtmp = m[,which(!grepl("^relabu_|_relabu$", names(m))), with = FALSE]
    mtmp[
        ,
        c(cov_cols, "taxonomy") := c(
            .SD[, cov_cols, with = FALSE],
            list(purrr::map(strsplit(taxonomy, "; ?"), function(x) paste0(tail(x,2), collapse = "; ")))
        )
    ][]
}

In [20]:
# kingdom
format_table(
    m[grep("d__", taxonomy),
      lapply(.SD, sum),
      by = list(sample, taxonomy = gsub(";p__.*", "", taxonomy))]
)

sample,taxonomy,coverage__singlem,coverage__sylph
<chr>,<list>,<dbl>,<dbl>
SRR8648366,d__Archaea,24.02,4.513
SRR8648366,d__Bacteria,3949.20,422.244


In [23]:
# phylum
format_table(
    m[!grepl("d__Archaea", taxonomy) & grepl(";p__", taxonomy),
      lapply(.SD, sum),
      by = list(sample, taxonomy = gsub(";c__.*", "", taxonomy))][coverage__sylph > 1]
)

sample,taxonomy,coverage__singlem,coverage__sylph
<chr>,<list>,<dbl>,<dbl>
SRR8648366,d__Bacteria; p__Actinobacteriota,2412.07,252.468
SRR8648366,d__Bacteria; p__Bacteroidota,361.75,38.146
SRR8648366,d__Bacteria; p__Firmicutes,82.91,33.306
SRR8648366,d__Bacteria; p__Firmicutes_A,123.84,44.409
SRR8648366,d__Bacteria; p__Myxococcota,10.13,1.609
SRR8648366,d__Bacteria; p__Proteobacteria,581.00,44.555
SRR8648366,d__Bacteria; p__Spirochaetota,3.81,2.314


In [25]:
# class
format_table(
    m[!grepl("d__Archaea", taxonomy) & grepl(";c__", taxonomy),
      lapply(.SD, sum),
      by = list(sample, taxonomy = gsub(";o__.*", "", taxonomy))][coverage__sylph > 1]
)

sample,taxonomy,coverage__singlem,coverage__sylph
<chr>,<list>,<dbl>,<dbl>
SRR8648366,p__Actinobacteriota; c__Actinomycetia,2269.85,251.125
SRR8648366,p__Bacteroidota; c__Bacteroidia,341.48,38.146
SRR8648366,p__Firmicutes; c__Bacilli,82.91,33.306
SRR8648366,p__Firmicutes_A; c__Clostridia,123.39,44.409
SRR8648366,p__Myxococcota; c__Polyangia,2.75,1.384
SRR8648366,p__Proteobacteria; c__Alphaproteobacteria,50.79,3.940
SRR8648366,p__Proteobacteria; c__Gammaproteobacteria,530.21,40.615
SRR8648366,p__Spirochaetota; c__Spirochaetia,3.75,2.314


In [26]:
# order
format_table(
    m[!grepl("d__Archaea", taxonomy) & grepl(";o__", taxonomy),
      lapply(.SD, sum),
      by = list(sample, taxonomy = gsub(";f__.*", "", taxonomy))][coverage__sylph > 1]
)

sample,taxonomy,coverage__singlem,coverage__sylph
<chr>,<list>,<dbl>,<dbl>
SRR8648366,c__Actinomycetia; o__Actinomycetales,1411.27,171.775
SRR8648366,c__Actinomycetia; o__Mycobacteriales,521.11,73.607
SRR8648366,c__Actinomycetia; o__Propionibacteriales,241.89,4.377
SRR8648366,c__Actinomycetia; o__Streptosporangiales,19.90,1.215
SRR8648366,c__Bacteroidia; o__Bacteroidales,59.50,30.037
SRR8648366,c__Bacteroidia; o__Flavobacteriales,164.17,8.109
SRR8648366,c__Bacilli; o__Acholeplasmatales,9.79,1.434
SRR8648366,c__Bacilli; o__Bacillales_D,11.21,1.829
SRR8648366,c__Bacilli; o__Erysipelotrichales,4.45,1.519


In [27]:
# family
format_table(
    m[!grepl("d__Archaea", taxonomy) & grepl(";f__", taxonomy),
      lapply(.SD, sum),
      by = list(sample, taxonomy = gsub(";g__.*", "", taxonomy))][coverage__sylph > 1]
)

sample,taxonomy,coverage__singlem,coverage__sylph
<chr>,<list>,<dbl>,<dbl>
SRR8648366,o__Actinomycetales; f__Beutenbergiaceae,88.61,2.483
SRR8648366,o__Actinomycetales; f__Brevibacteriaceae,46.74,20.681
SRR8648366,o__Actinomycetales; f__Cellulomonadaceae,34.79,1.086
SRR8648366,o__Actinomycetales; f__Dermabacteraceae,74.48,3.965
SRR8648366,o__Actinomycetales; f__Dermatophilaceae,979.46,140.619
SRR8648366,o__Actinomycetales; f__Microbacteriaceae,82.61,1.357
SRR8648366,o__Mycobacteriales; f__Mycobacteriaceae,310.89,69.718
SRR8648366,o__Mycobacteriales; f__Pseudonocardiaceae,54.79,3.696
SRR8648366,o__Propionibacteriales; f__Propionibacteriaceae,92.21,4.377


In [28]:
# How well does genus level rescue some of the missing genomes? First need to remake the table with genus level, annoying since kraken profiles are filled, when the rest aren't.
format_table(
    m[!grepl("d__Archaea", taxonomy) & grepl(";g__", taxonomy),
      lapply(.SD, sum),
      by = list(sample, taxonomy = gsub(";s__.*", "", taxonomy))][coverage__sylph > 1]
)

sample,taxonomy,coverage__singlem,coverage__sylph
<chr>,<list>,<dbl>,<dbl>
SRR8648366,f__Beutenbergiaceae; g__Ruania,74.01,2.136
SRR8648366,f__Brevibacteriaceae; g__Brevibacterium,37.09,20.681
SRR8648366,f__Dermabacteraceae; g__Brachybacterium,73.72,3.965
SRR8648366,f__Dermatophilaceae; g__F2B08,103.62,57.276
SRR8648366,f__Dermatophilaceae; g__Janibacter,132.76,2.683
SRR8648366,f__Dermatophilaceae; g__Ornithinimicrobium,184.53,80.175
SRR8648366,f__Microbacteriaceae; g__Gulosibacter,2.31,1.037
SRR8648366,f__Mycobacteriaceae; g__Corynebacterium,74.19,54.504
SRR8648366,f__Mycobacteriaceae; g__Dietzia,23.00,4.428


In [33]:
# Species level
format_table(m[!grepl("d__Archaea", taxonomy) & grepl(";s__", taxonomy)][coverage__sylph > 1])

sample,taxonomy,coverage__singlem,coverage__sylph
<chr>,<list>,<dbl>,<dbl>
SRR8648366,g__Ruania; s__Ruania albidiflava,23.60,2.136
SRR8648366,g__Brevibacterium; s__Brevibacterium epidermidis,0.00,1.355
SRR8648366,g__Brevibacterium; s__Brevibacterium intestinavium,27.17,18.178
SRR8648366,g__Brachybacterium; s__Brachybacterium massiliense,10.18,2.379
SRR8648366,g__Brachybacterium; s__Brachybacterium paraconglomeratum,5.26,1.586
SRR8648366,g__F2B08; s__F2B08 sp012729695,7.12,4.798
SRR8648366,g__F2B08; s__F2B08 sp012838445,79.93,52.478
SRR8648366,g__Janibacter; s__Janibacter corallicola,55.36,2.683
SRR8648366,g__Ornithinimicrobium; s__Ornithinimicrobium sp003577095,56.63,80.175
